# 03 — Wyjaśnialność (SHAP, reguły ESI, Ollama)

Wyjaśnienia dla pacjentów ze zbioru val na modelu z zapisanego runa (`runs/`).
Ten sam pipeline w skrypcie: `python scripts/05_explain.py`.
SHAP: TreeSHAP (XGBoost / LightGBM / RF), natywne SHAP CatBoost, liniowy (regresja
logistyczna), wkłady termów (EBM).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data import load_split
from src.data.prepare import META_COLS
from src.ensemble import list_runs, load_run
from src.evaluation.metrics import decide
from src.explain import (OllamaMedicalExplainer, SHAPTriageExplainer, check_consistency,
                         explain_rule_decision, load_background, rule_based_triage,
                         shap_summary_plot, shap_waterfall_plot)
from src.models import load_model
from src.utils.config import CLASS_NAMES, CLASS_NAMES_PL

In [ ]:
MODEL, FEATURE_SET = 'xgboost', 'triage'
table = list_runs()
table = table[(table['model'] == MODEL) & (table['feature_set'] == FEATURE_SET)]
names = [n for n in table['run'] if (load_run(n).path / 'model.joblib').exists()]
run = load_run(names[-1])                          # najnowszy run z zapisanym modelem
model = load_model(run.path / 'model.joblib')     # klasa wg nazwy modelu w artefakcie
features = model.feature_names
print(run.name, '|', model.feature_set, '|', len(features), 'cech')

In [ ]:
df = load_split('val', columns=META_COLS + features)
df = df.sample(min(2_000, len(df)), random_state=42)
X = df[features]
proba = model.predict_proba(X)
df['pred_argmax'] = decide(proba, 'argmax')
df['pred_cost'] = decide(proba, 'cost')
pd.crosstab(df['esi'], df['pred_cost'] + 1, rownames=['ESI prawdziwy'], colnames=['ESI (kosztowa)'])

## SHAP — globalnie (wszystkie poziomy i beeswarm dla ESI 1)

In [ ]:
method = SHAPTriageExplainer.method_for(model)
background = load_background(features) if method in ('linear', 'permutation') else None
explainer = SHAPTriageExplainer(model, background_data=background).fit()

sample = X.sample(min(300, len(X)), random_state=0)
shap_values = explainer.explain_dataset(sample)
shown = explainer.feature_frame(sample)
shap_summary_plot(shap_values, shown, show=True)
shap_summary_plot(shap_values, shown, class_idx=0, show=True)

## SHAP — pacjent wysokiej pilności (waterfall)

In [ ]:
patient = X.loc[[df.index[df['esi'] <= 2][0]]]
explained_idx = int(decide(model.predict_proba(patient), 'cost')[0])
exp = explainer.explain_patient(patient, class_idx=explained_idx)

print(f"Prawdziwy: ESI {int(df.loc[patient.index[0], 'esi'])} | wyjaśniany: {CLASS_NAMES_PL[explained_idx]}")
print({k: round(v, 3) for k, v in exp['probabilities'].items()})
for f in exp['top_features_for'][:5]:
    print(f"  + {f['feature']}: {f['shap_value']:+.3f} (wartość={f['patient_value']})")
for f in exp['top_features_against'][:3]:
    print(f"  - {f['feature']}: {f['shap_value']:+.3f} (wartość={f['patient_value']})")
shap_waterfall_plot(explainer, patient, class_idx=explained_idx, show=True)

## Reguły ESI (parametry życiowe, punkty A / B / D)

Reguły wskazują ESI 1 (zagrożenie życia) lub ESI 2 (wysokie ryzyko / strefa zagrożenia).
ESI 3–5 zależy od przewidywanych zasobów — z samych parametrów nie da się go ustalić.

In [ ]:
patient_dict = df.loc[patient.index[0]].to_dict()
rule = rule_based_triage(patient_dict)
print(explain_rule_decision(rule))
print()
print(check_consistency(explained_idx, rule['suggested_class_idx'])['verdict'])

In [ ]:
checks = [rule_based_triage(r) for r in X.to_dict('records')]
rule_idx = pd.Series([c['suggested_class_idx'] for c in checks], index=df.index, dtype='float')
flagged = rule_idx.notna() & (df['pred_cost'] > rule_idx)
print(f'Reguły wskazują ESI 1–2: {rule_idx.notna().mean():.1%} pacjentów | '
      f'model (kosztowa) mniej pilny niż reguły: {flagged.mean():.1%}')
labels = rule_idx.map({0.0: 'ESI 1', 1.0: 'ESI 2'}).fillna('ESI 3–5').rename('reguły')
display(flagged.groupby(df['vitals_complete']).mean().rename('model mniej pilny niż reguły').round(3))
pd.crosstab(labels, df['esi'].rename('ESI prawdziwy'), normalize='index').round(3)

## Wyjaśnienie medyczne LLM (opcjonalnie, wymaga Ollama)

In [ ]:
ollama = OllamaMedicalExplainer()
if ollama.is_available():
    print(ollama.explain(
        patient_data=patient_dict,
        predicted_class=CLASS_NAMES[explained_idx],
        predicted_class_pl=CLASS_NAMES_PL[explained_idx],
        probabilities=exp['probabilities'],
        shap_explanation=exp,
        rule_check=rule,
    ))
else:
    print('Ollama niedostępne. Aby aktywować:\n  ollama serve\n  ollama pull mistral')